Rewards
========

Rewards Interface
-----------------

The `Rewards` interface has the following methods:

* `step_reward`: Handles end-of-step-reward for a particular agent.
* `end_of_episode_reward`: Handles end-of-episode reward for a particular agent.
* `cumulate`: Cumulate multiple rewards to one.
* `empty`: Return empty initial value neutral for the cumulation.
* `normalize`: Return normalized cumulated rewards (can be `None` for some rewards).

The env calls `step_reward(agent, agent_transition_data, distance_map, elapsed_steps)` for each agent at the end of each step, after it has updated the
agent's entry points, speed and distance: `agent.current_entry_point` is the agent's entry point after the step and `agent.old_entry_point` the one
before. `agent_transition_data` (`AgentTransitionData`) tells a custom reward what happened to the agent in this step:

| field                                                 | description                                                                                                                                                                          |
|:------------------------------------------------------|:-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `action`                                              | The action as given to `step()` (`DO_NOTHING` if none was given), before any validity check.                                                                                         |
| `action_valid`                                        | Whether the action is valid: always inside a cell; at the end of a cell only if it leads to a valid transition (an invalid action is today only possible facing a symmetric switch). |
| `resource_check`                                      | Whether the resource check (`env.resource_check`) granted the agent's move, i.e. no conflict with another agent. The move is allowed if `action_valid and resource_check`.           |
| `speed`, `distance`                                   | Speed and distance before the step, `None` while the agent was off the map.                                                                                                          |
| `candidate_speed`, `candidate_distance`               | Speed and distance the env computed for after the step; on the step the agent enters the map, `candidate_speed` is its entry speed.                                                  |
| `candidate_entry_point`, `candidate_next_entry_point` | Entry points the agent moves to if its move is allowed.                                                                                                                              |
| `is_cell_exit`                                        | Whether the agent, at its speed before the step, reaches the end of its cell in this step (speed > 0 and distance + speed ≥ 1); `True` while off the map.                            |
| `done`                                                | Whether the agent had already arrived at its target before this step.                                                                                                                |

These fields do not depend on `agent.state`, so a reward computed from them works with and without the state machine (see [Agent](agent)). For
example, `BaseDefaultRewards` charges a forced stop only for a train that was moving before the step (`speed > 0`) and is stopped by the env, and it
tells the two causes apart by `action_valid` (`INVALID_ACTION`) and `resource_check` (`COLLISION`).

`end_of_episode_reward(agent, distance_map, elapsed_steps)` is called once per agent when the episode ends (all agents done or `max_episode_steps`
reached).

Default Rewards
---------------

The Flatland `DefaultRewards` scoring function is designed to capture key operational metrics such as punctuality, efficiency in responding to disruptions, and
safety.
Punctuality and schedule adherence are rewarded based on the difference between actual and target arrival and departure times at each stop respectively,
as well as penalties for intermediate stops not served or even journeys not started.
Safety measures are implemented as penalties for env-forced stops (collisions and invalid actions) which are directly proportional to the train’s speed at
impact, ensuring that high-speed operations are managed with extra caution.

More formally, the score

$$
\begin{aligned}
S &= \sum_{i=1}^N g_i
\end{aligned}
$$

is the sum of all rewards $g_i$ of agent $i$. The reward $g_i$ of an agent consists of a reward at the end of the episode as well as at each timestep $t.$
Per-timestep rewards only consist of a penalty if the environment forces the train to stop, i.e. the train goes from MOVING to STOPPED without a stop action
bringing it to zero speed: either it wants to enter a cell another train already occupies (potential collision), or its action is invalid. This collision penalty is
proportional to the current speed $v(t)$ of the train. The reward $g_i$ of an agent $i$ is

$$
\begin{aligned}
g_i = &
% delay at target:
\underbrace{\mathrm{A}_J \cdot  \min \{\alpha_J - a_J,0\}}_{\text{delay at target}}  
% journey not started:
+ \underbrace{(1 - \Delta_1) \cdot \phi \cdot (-(p + \pi))}_{\text{journey not started}}
% target not reached:
+ \underbrace{(1 - \mathrm{A}_J) \cdot \min\{-\nu, \alpha_J - T - r\}}_{\text{target not reached}}\\
& + \sum_{j=2}^{J-1} \Big[
% intermediate late arrival
\underbrace{\mathrm{A}_j \cdot \alpha \cdot \min \{\alpha_j - a_j,0\}}_{\text{late arrival}}
% intermediate stop not served
+ \underbrace{(1 - \mathrm{A}_j) \cdot \mu \cdot (-1)}_{\text{stop not served}}
% intermediate early departure
+ \underbrace{\Delta_j \cdot \delta \cdot \min \{ d_j - \delta_j, 0 \}}_{\text{early departure}} \Big] \\
% collision
& + \sum_{t=1}^T  \underbrace{\mathrm{K}_t \cdot \kappa \cdot (- v(t))}_{\text{collision}} \ ,
\end{aligned}
$$

where $J$ is the number of stops (including the departure at the start, as well as the target) and $T$ is the maximum number of timesteps of the episode (`max_episode_steps`).
The symbols are described in the table below.

|                              | penalty factor <br/>($\geq 0$) | event <br/> $\in \{0,1\}$ | scheduled  | actual | description                                                                                                                                                                                                                     |
|:-----------------------------|--------------------------------|---------------------------|------------|-----|---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| delay at target              | 1                              | $\mathrm{A}_J$            | $\alpha_J$ | $a_J$ | $\alpha_J$ latest arrival and $a_J$ actual arrival at target $J$                                                                                                                                                                |
| journey not started          | $\phi$, $\pi$                  | $1-\Delta_1$              | $p$        |     | cancellation factor $\phi$ and buffer $\pi$,    <br/> $p$ is the shortest path from start to target                                                                                                                             |
| target not reached           | 1                              | $1-\mathrm{A}_J$          | $\alpha_J$ | $r$ | projected delay $\alpha_J - T - r$ at the end of the episode, with time $r$ remaining on shortest path to target with maximum speed for corresponding train category; at least the minimum penalty $\nu$ for target not reached |
| intermediate late arrival    | $\alpha$                       | $\mathrm{A}_j$            | $\alpha_j$ | $a_j$ | latest arrival $\alpha_j$, actual arrival time $a_j$    <br/> at intermediate stop $j=2,\ldots,J-1$                                                                                                                             |
| intermediate stop not served | $\mu$                          | $1-\mathrm{A}_j$          |            |     | intermediate stop $j$ not served (not stopped at), $j=2,\ldots,J-1$                                                                                                                                                             |
| intermediate early departure | $\delta$                       | $\Delta_j$                | $\delta_j$ | $d_j$ | earliest departure from stop $j$, actual departure time $d_j$ <br/> at intermediate stop $j=2,\ldots,J-1$                                                                                                                       |
| collision                    | $\kappa$                       | $\mathrm{K}_t$            |            | $v(t)$ | collision or invalid action forcing a stop at time $t$ with speed $v(t)$                                                                                                                                                       |

The events used in the table are:

| event          | description                                                                                                     |
|:---------------|:----------------------------------------------------------------------------------------------------------------|
| $\mathrm{A}_J$ | arrived at target $J$                                                                                           |
| $\Delta_1$     | departed from start                                                                                             |
| $\mathrm{A}_j$ | stopped at intermediate stop $j$ (at one of its halting cells); running through without stopping does not count |
| $\Delta_j$     | departed from intermediate stop $j$                                                                             |
| $\mathrm{K}_t$ | forced stop at time $t$ (collision or invalid action)                                                           |

`BaseDefaultRewards` returns the rewards per category (`DefaultPenalties`), `DefaultRewards` their sum. Since `flatland 5.0.0`, a forced stop is tracked as
`COLLISION` if the train conflicts with another train, and as `INVALID_ACTION` if its action is invalid (today only possible facing a symmetric switch); before,
both were tracked as `COLLISION`. Both use the same factor $\kappa$, so the totals are unchanged. A blocked train that keeps trying to move is charged at each
denied attempt.

The categories correspond to the terms of $g_i$ as follows:

| category (`DefaultPenalties`)        | term                             |
|:-------------------------------------|:---------------------------------|
| `COLLISION`, `INVALID_ACTION`        | collision                        |
| `TARGET_LATE_ARRIVAL`                | delay at target                  |
| `CANCELLATION`                       | journey not started              |
| `TARGET_NOT_REACHED`                 | target not reached (incl. $\nu$) |
| `INTERMEDIATE_LATE_ARRIVAL`          | late arrival                     |
| `INTERMEDIATE_NOT_SERVED`            | stop not served                  |
| `INTERMEDIATE_EARLY_DEPARTURE`       | early departure                  |
| `TARGET_NOT_REACHED_MINIMUM_PENALTY` | not used, always 0               |

The scheduled and actual values used in the table are:

| scheduled  | actual | description                                                             |
|:-----------|:-------|:------------------------------------------------------------------------|
| $\alpha_J$ | $a_J$  | latest / actual arrival at target                                       |
| $\alpha_j$ | $a_j$  | latest / actual arrival at intermediate stop $j$                        |
| $\delta_j$ | $d_j$  | earliest / actual departure from intermediate stop $j$ (see below)      |
| $p$        |        | time on the shortest path from the scheduled start to target            |
|            | $r$    | time remaining on the shortest path to target at the end of the episode |
|            | $v(t)$ | speed at time $t$                                                       |

The constants used in the table are the penalty factors and parameters of `DefaultRewards` (with their defaults) and the episode length:

| constant   | parameter                                     | default | description                                     |
|:-----------|:----------------------------------------------|:--------|:------------------------------------------------|
| $\phi$     | `cancellation_factor`                         | 1.0     | penalty factor for journey not started          |
| $\pi$      | `cancellation_time_buffer`                    | 0.0     | time buffer for journey not started             |
| $\nu$      | `target_not_reached_minimum_penalty`          | 0.0     | minimum penalty for target not reached          |
| $\alpha$   | `intermediate_late_arrival_penalty_factor`    | 0.2     | penalty factor for intermediate late arrival    |
| $\mu$      | `intermediate_not_served_penalty`             | 1.0     | penalty for intermediate stop not served        |
| $\delta$   | `intermediate_early_departure_penalty_factor` | 0.5     | penalty factor for intermediate early departure |
| $\kappa$   | `collision_factor`                            | 0.0     | penalty factor for collision                    |
| $T$        | `max_episode_steps` of the env                |         | maximum number of time steps of the episode     |

Note that the simulation enforces that agents cannot start earlier than $\delta_1$ at their start. On the other hand, early departure at intermediate stops is
not enforced by the simulation, but will be penalized by the rewards function.
Also note that order of intermediate stops is also not enforced by the simulation in case of overlapping time windows.

The actual departure $d_j$ from a stop is the step in which the agent starts moving after its last halt there, i.e. the step after its last step with speed 0
at the stop:

- A halt is speed 0 after a step for any reason: `STOP_MOVING`, a denied crossing (invalid action or occupied cell), or a malfunction. An agent that waits out a
  malfunction at a stop and starts at $\delta_j$ or later departs on time.
- An agent that creeps forward within the stop and halts again departs in the step after its second halt: creeping forward is not a departure.
- An agent that passes the stop without halting departs in the step in which it enters the next cell.

An agent that starts from a halt reaches the next cell only once it has covered the rest of its cell, up to $1/v_{max}$ steps later (see
[Speed Profiles](speed_profiles)). The departure is the start, not that later step. For example, an agent with max speed $1/4$ halts at stop $j$ with
$\delta_j = 12$, creeps forward in step 7, halts again, and starts in step 10: it reaches the next cell in step 12, but $d_j = 10$, so it is penalized for an early
departure of 2 steps, $\delta \cdot (10 - 12) = -1$ with the default $\delta = 0.5$. `PunctualityRewards` uses the same departure.

The plot shows this run: the speed after each step (blue), the distance travelled (brown) with the cells of the stop (band), the state, the stop's earliest
departure (dashed) and the departure the rewards book (purple).

In [ ]:
from fractions import Fraction

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import to_rgb

from flatland.envs.grid.rail_env_grid import RailEnvTransitions
from flatland.envs.observations import FullEnvObservation
from flatland.envs.rail_env import RailEnv
from flatland.envs.rail_env_action import RailEnvActions
from flatland.envs.rail_generators import rail_from_grid_transition_map
from flatland.envs.rail_grid_transition_map import RailGridTransitionMap
from flatland.envs.rail_trainrun_data_structures import Waypoint
from flatland.envs.rewards import DefaultRewards
from flatland.envs.timetable_utils import Line, Timetable

# the plot is the one of the Speed Profiles page, with the departure from a stop added
SPEED_COLOR, DISTANCE_COLOR, INK, MUTED, GRID = "#1d4ed8", "#b45309", "#1f2937", "#6b7280", "#e5e7eb"
EARLIEST_DEPARTURE_COLOR, DEPARTURE_COLOR, STOP_COLOR = "#dc2626", "#7c3aed", "#ede9fe"
# state: (fill, faded out to the right, as a state holds from its first elapsed_steps up to just before the next state;
# start line, at the first elapsed_steps of the state)
STATE_COLORS = {"WAITING": ("#e5e7eb", "#6b7280"), "READY_TO_DEPART": ("#fde68a", "#d97706"),
                "MOVING": ("#bbf7d0", "#16a34a"), "STOPPED": ("#fecaca", "#dc2626"), "DONE": ("#ddd6fe", "#7c3aed")}


def plot_departure(rows, stop_cells, earliest_departure, departure, penalty, labels, title):
    """Speed (left axis) and distance travelled (right axis) per elapsed_steps, as steps: a value set by the step to
    elapsed_steps = k holds from k up to just before k+1; a fine line interpolates it during the step. Off the map, speed
    and distance are None, shown as 0. The band marks the cells of the stop, the dashed line its earliest departure and
    the solid line the departure the rewards book.
    rows: per elapsed_steps from 0, (elapsed_steps, state, speed, distance travelled).
    labels: (k, action given at elapsed_steps = k, i.e. into the step k -> k+1, and its comment, y of the arrow tip,
    y of the text, both on the speed axis)."""
    x_max = len(rows) - 1
    xs = list(range(x_max + 1))
    speed = [float(row[2] or 0) for row in rows]
    travelled = [float(row[3] or 0) for row in rows]
    fig, (ax_s, ax_states) = plt.subplots(2, 1, figsize=(9, 6.7), sharex=True,
                                          gridspec_kw=dict(height_ratios=[6, 1.0], hspace=0.08))
    ax_d = ax_s.twinx()
    # the stop's cells, drawn on the speed axis below everything (distance 3 is at speed 1, see the y limits below)
    ax_s.axhspan(stop_cells[0] / 3, stop_cells[1] / 3, color=STOP_COLOR, alpha=0.8, zorder=0)
    ax_s.text(x_max - 0.1, sum(stop_cells) / 6, "stop", ha="right", va="center", fontsize=8, color=MUTED)
    line_s, = ax_s.step(xs, speed, where="post", color=SPEED_COLOR, linewidth=2, label="speed (cells per step, left)")
    line_d, = ax_d.step(xs, travelled, where="post", color=DISTANCE_COLOR, linewidth=2,
                        label="distance travelled (cells, right)")
    ax_s.plot(xs, speed, color=SPEED_COLOR, linewidth=0.8)
    ax_d.plot(xs, travelled, color=DISTANCE_COLOR, linewidth=0.8)
    # increments, right of each jump (none at the right edge, where the label would cross the axis)
    for ax, values, color in ((ax_s, speed, SPEED_COLOR), (ax_d, travelled, DISTANCE_COLOR)):
        for k in range(1, x_max):
            if values[k] != values[k - 1]:
                increment = Fraction(values[k] - values[k - 1]).limit_denominator(8)
                ax.text(k + 0.06, (values[k - 1] + values[k]) / 2, f"{'+' if increment > 0 else ''}{increment}", ha="left",
                        va="center", fontsize=7, color=color)
    line_ed = ax_s.axvline(earliest_departure, color=EARLIEST_DEPARTURE_COLOR, linestyle="--", linewidth=1.5,
                           label=f"earliest departure = {earliest_departure}")
    line_dep = ax_s.axvline(departure, color=DEPARTURE_COLOR, linewidth=3, alpha=0.8, label=f"departure = {departure}")
    verdict = f"early by {earliest_departure - departure}: penalty {penalty:g}" if departure < earliest_departure else "on time"
    ax_s.text(departure + 0.1, 1.42, f"departure = {departure}\n{verdict}", color=DEPARTURE_COLOR, fontsize=9, va="top",
              fontweight="bold")
    ax_s.set_xlim(0, x_max)
    ax_s.set_ylim(0, 1.5)
    ax_d.set_ylim(0, 4.5)  # speed 1 and distance 3 at the same height
    ax_s.set_yticks([0, 0.25, 0.5, 0.75, 1])
    ax_d.set_yticks(range(5))
    ax_s.set_ylabel("speed (cells per step)", color=INK)
    ax_d.set_ylabel("distance travelled (cells)", color=INK)
    for ax in (ax_s, ax_d):
        ax.spines["top"].set_visible(False)
        ax.tick_params(colors=INK, labelsize=9)
    ax_s.spines["left"].set_color(SPEED_COLOR)
    ax_s.spines["right"].set_visible(False)
    ax_d.spines["left"].set_visible(False)
    ax_d.spines["right"].set_color(DISTANCE_COLOR)
    ax_s.grid(axis="x", color=GRID, linewidth=0.8)
    ax_s.set_axisbelow(True)
    for k, text, y_tip, y_text in labels:
        ax_s.annotate(text, xy=(k, y_tip), xytext=(k + 0.06, y_text), fontsize=8.5, color=INK, ha="left", va="bottom",
                      arrowprops=dict(arrowstyle="->", color=MUTED, linewidth=1, shrinkA=0, shrinkB=2, relpos=(0, 0)),
                      bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor=GRID, linewidth=0.8))
    fig.legend(handles=[line_s, line_d, line_ed, line_dep], loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=4,
               frameon=False, fontsize=9, labelcolor=INK)
    ax_s.set_title(title, color=INK, fontsize=11, loc="left")

    # state after each step, as a bar below the x axis
    states = [row[1] for row in rows]
    starts = [k for k in range(len(states)) if k == 0 or states[k] != states[k - 1]]
    for i, start in enumerate(starts):
        end = ([*starts[1:], x_max])[i]
        fill, edge = STATE_COLORS[states[start]]
        gradient = np.dstack([np.broadcast_to(to_rgb(fill), (1, 64, 3)), np.linspace(1, 0.1, 64).reshape(1, 64, 1)])
        ax_states.imshow(gradient, extent=(start, end, 0.06, 0.94), aspect="auto", interpolation="bilinear", zorder=1)
        ax_states.plot([start, start], [0.06, 0.94], color=edge, linewidth=2.5, zorder=2)
        short = end - start < 2
        ax_states.text((start + end) / 2, 0.5, states[start].replace("_TO_", "_TO_\n") if short else states[start],
                       ha="center", va="center", fontsize=6.5 if short else 8, color=INK, linespacing=1.0, zorder=3)
    ax_states.axvline(earliest_departure, color=EARLIEST_DEPARTURE_COLOR, linestyle="--", linewidth=1.5, zorder=4)
    ax_states.axvline(departure, color=DEPARTURE_COLOR, linewidth=3, alpha=0.8, zorder=4)
    ax_states.set_ylim(0, 1)
    ax_states.set_yticks([0.5], ["state"], fontsize=8)
    ax_states.tick_params(axis="y", length=0)
    for side in ("top", "left", "right"):
        ax_states.spines[side].set_visible(False)
    ax_states.set_xticks(xs)
    ax_states.tick_params(colors=INK, labelsize=9)
    ax_states.set_xlabel("elapsed_steps", color=INK)
    plt.show()


def run_corridor(max_speed, stop, earliest_departure_at_stop, actions, steps):
    """One agent on a straight east-bound track of 10 cells from (0,1) via the intermediate stop to its target (0,8),
    with the default rewards. actions: the action into the step to elapsed_steps = k, MOVE_FORWARD if not given.
    Returns the values after reset() and after each step, as (elapsed_steps, state, speed, distance travelled), the
    departure from the stop the rewards book, and the early departure penalty."""
    n_cells = 10
    transitions = RailEnvTransitions()
    straight = transitions.rotate_transition(transitions.transition_list[1], 90)
    dead_end = transitions.transition_list[7]
    rail = RailGridTransitionMap(width=n_cells, height=1, transitions=transitions)
    rail.grid = np.array([[transitions.rotate_transition(dead_end, 270)] + [straight] * (n_cells - 2)
                          + [transitions.rotate_transition(dead_end, 90)]], dtype=np.uint16)
    start, target = ((0, 1), 1), ((0, n_cells - 2), 1)
    waypoints = [[Waypoint(*start)], [Waypoint(*stop)], [Waypoint(*target)]]
    rewards = DefaultRewards()
    env = RailEnv(width=n_cells, height=1, rail_generator=rail_from_grid_transition_map(rail),
                  line_generator=lambda *args, **kwargs: Line(agent_waypoints={0: waypoints}, agent_speeds=[max_speed]),
                  timetable_generator=lambda *args, **kwargs: Timetable(
                      earliest_departures=[[1, earliest_departure_at_stop, None]], latest_arrivals=[[None, 40, 60]],
                      max_episode_steps=60),
                  number_of_agents=1, obs_builder_object=FullEnvObservation(), rewards=rewards)
    env.reset()
    agent = env.agents[0]

    def row():
        entry_point = agent.current_entry_point
        travelled = None if entry_point is None else entry_point[0][1] - start[0][1] + agent.speed_counter.distance
        return env._elapsed_steps, agent.state.name, agent.speed_counter.speed, travelled

    rows, penalty = [row()], 0
    for k in range(1, steps + 1):
        _, step_rewards, _, _ = env.step({0: actions.get(k, RailEnvActions.MOVE_FORWARD)})
        rows.append(row())
    # run to the end of the episode for the end-of-episode rewards
    while not env.dones["__all__"]:
        env.step({0: RailEnvActions.MOVE_FORWARD})
    departure = rewards._proxy.departures[0][stop][0]
    penalty = rewards.intermediate_early_departure_penalty_factor * min(departure - earliest_departure_at_stop, 0)
    return rows, departure, penalty

In [ ]:
# max speed 1/4; the stop is the cell at distance travelled 1-2, its earliest departure is 12. The agent enters the stop in
# step 5, brakes (speed 0 from step 6), creeps forward in step 7, halts again (steps 8 and 9) and starts in step 10.
STOP, STOP_MOVING = ((0, 2), 1), RailEnvActions.STOP_MOVING
rows, departure, penalty = run_corridor(max_speed=Fraction(1, 4), stop=STOP, earliest_departure_at_stop=12,
                                        actions={6: STOP_MOVING, 8: STOP_MOVING, 9: STOP_MOVING}, steps=16)
assert [row[2] for row in rows[5:13]] == [Fraction(1, 4), 0, Fraction(1, 4), 0, 0, Fraction(1, 4), Fraction(1, 4), Fraction(1, 4)]
assert rows[12][3] == 2, "the agent reaches the next cell in step 12"
assert (departure, penalty) == (10, -1.0), (departure, penalty)
plot_departure(rows, stop_cells=(1, 2), earliest_departure=12, departure=departure, penalty=penalty,
               labels=[(5, "STOP_MOVING: halts in the stop", 0, 0.62),
                       (6, "MOVE_FORWARD: creeps 1/4 cell", 0.25, 0.86),
                       (7, "STOP_MOVING: halts again", 0, 1.1),
                       (9, "MOVE_FORWARD: starts", 0.25, 0.72)],
               title="Departure from a stop: the step after the last halt\nmax speed 1/4, earliest departure from the stop = 12")

Since `flatland 5.0.0`, the departure is the start from the last halt. Up to `flatland 4.3.0`, it was the step in which the agent entered the next cell. At max
speed 1 this was the step in which it started, as it left its cell in that step; for slower agents it was later.

To compare results between environments of different dimensions and different number of agents, the reward can be normalized, such that the normalized reward is in the range $[0.0, 1.0]$. For each agent there is a maximum penalty set at ```- max_episode_steps```. This guarantees normalization regardless of the parametrization of the rewards.

```{admonition} Code reference
The reward is calculated in [envs/rewards.py](https://github.com/flatland-association/flatland-rl/blob/main/flatland/envs/rewards.py)
```

Punctuality Rewards
-------------------

`PunctualityRewards` is `n_stops_on_time / n_stops`.

An agent is deemed not punctual at a stop if it arrives too late, departs too early (departure as defined for `DefaultRewards` above) or does not serve the
stop at all.

If an agent is punctual at a stop, n_stops_on_time is increased by 1.

> This feature was introduced in [4.2.2](https://github.com/flatland-association/flatland-rl/pull/272)

Basic Multi-Objective Rewards
-----------------------------

For illustration purposes, a basic MORL (Multi-Objective Reinforcement Learning) `BasicMultiObjectiveRewards` is implemented. It is composed of 3 components:

- default score
- energy efficiency: `- square (speed/max_speed).`
- smoothness: `- square (speed differences)`

> This feature was introduced in [4.2.1](https://github.com/flatland-association/flatland-rl/pull/269)